## Question 10 - *How often does it happen that the resources of a machine are over-committed?*

In [3]:
import sys
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F
import time
from pyspark.storagelevel import StorageLevel
import pandas as pd

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

task_events_1 = sc.textFile("./data/task_events/part-00060-of-00500.csv.gz")
task_events_2 = sc.textFile("./data/task_events/part-00061-of-00500.csv.gz")
task_events_3 = sc.textFile("./data/task_events/part-00062-of-00500.csv.gz")
task_events_4 = sc.textFile("./data/task_events/part-00063-of-00500.csv.gz")
task_events_4 = sc.textFile("./data/task_events/part-00064-of-00500.csv.gz")

# Get machine capacities with proper error handling
machine_capacities = (
    sc.textFile("./data/machine_events/part-00000-of-00001.csv.gz")
    .map(lambda line: line.split(","))     # converting strings into fields
    .filter(lambda x: x[2] in ['0', '2'])  # keep only ADD or UPDATE
    .filter(lambda x: x[4] != '' and x[5] != '')  # keep only events that have values for CPU and memory
    .map(lambda x: ( x[1], (int(x[0]), float(x[4]), float(x[5]) ))) # ( machine_id, (timestamp, cpu_capacity, memory_capacity) )
    .reduceByKey(lambda a, b: a if a[0] > b[0] else b)  # removing the duplicates by only keeping the most recent value
    .mapValues(lambda x: (x[1], x[2]))  # keep only (cpu_capacity, mem_capacity)
)

print("\n" + "="*80)
machine_count = machine_capacities.count()
print(f"Total machines with capacity info: {machine_count}")
capacities_dict = dict(machine_capacities.collect()) # dictionary for faster lookup

scheduled_by_machine = (
    task_events_1.union(task_events_2).union(task_events_3).union(task_events_4)
    .map(lambda line: line.split(","))
    .filter(lambda x: x[5] == '1')      # keep only SCHEDULE events
    .filter(lambda x: x[9] != '' and x[10] != '')   # removing rows with missing values
    .map(lambda x: (x[4], (float(x[9]), float(x[10])))) # ( machine_id, (cpu_request, mem_request) )
    .groupByKey()
)

scheduled_machine_count = scheduled_by_machine.count()
print(f"Machines with scheduled tasks: {scheduled_machine_count}")

# to check overcommitment for a given machine
def check_overcommitment(machine_id, requests, capacities_dict):
    if machine_id not in capacities_dict:
        return None
    
    cpu_capacity, mem_capacity = capacities_dict[machine_id]
    requests_list = list(requests)
    
    # Sum all requests
    total_cpu = sum([r[0] for r in requests_list])
    total_mem = sum([r[1] for r in requests_list])
    
    # Check over-commitment
    cpu_overcommit = total_cpu > cpu_capacity
    mem_overcommit = total_mem > mem_capacity
    
    return {
        'machine_id': machine_id,
        'cpu_capacity': cpu_capacity,
        'cpu_overcommit': cpu_overcommit,
        'mem_capacity': mem_capacity,
        'mem_overcommit': mem_overcommit,
        'overcommitted': cpu_overcommit or mem_overcommit
    }

# applying overcommitment check
overcommit_results = []
for machine_id, requests in scheduled_by_machine.collect():     # where request = (cpu_request, mem_request)
    result = check_overcommitment(machine_id, requests, capacities_dict)
    if result:
        overcommit_results.append(result)

# converting to DataFrame
df = pd.DataFrame(overcommit_results)

print("\n" + "="*80)
print("OVERCOMMITMENT RESULTS")

total_machines = len(df)
cpu_overcommit_count = df['cpu_overcommit'].sum()
mem_overcommit_count = df['mem_overcommit'].sum()

print(f"\nTotal machines analyzed: {total_machines}")

print(f"CPU Over-commitment -> Machines: {cpu_overcommit_count} ({cpu_overcommit_count/total_machines*100:.2f}%)")

print(f"Memory Over-commitment -> Machines: {mem_overcommit_count} ({mem_overcommit_count/total_machines*100:.2f}%)")


Total machines with capacity info: 12583
Machines with scheduled tasks: 12157

OVERCOMMITMENT RESULTS

Total machines analyzed: 12157
CPU Over-commitment -> Machines: 5510 (45.32%)
Memory Over-commitment -> Machines: 5420 (44.58%)
